Dataset embeds

In [1]:
import kagglehub

path = kagglehub.dataset_download("seddiktrk/cafa6-protein-embeddings-esm2")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/cafa6-protein-embeddings-esm2


In [2]:
import os
print(os.listdir("/kaggle/input"))

['cafa-6-protein-function-prediction', 'cafa6-protein-embeddings-esm2']


In [3]:
import os, re, json
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler

In [4]:
COMP_DIR = "/kaggle/input/cafa-6-protein-function-prediction"
TRAIN_DIR = os.path.join(COMP_DIR, "Train")
TEST_DIR  = os.path.join(COMP_DIR, "Test")

TRAIN_TERMS = os.path.join(TRAIN_DIR, "train_terms.tsv")
TRAIN_FASTA = os.path.join(TRAIN_DIR, "train_sequences.fasta")
TEST_FASTA  = os.path.join(TEST_DIR,  "testsuperset.fasta")
OBO_PATH    = os.path.join(TRAIN_DIR, "go-basic.obo")

EMB_DIR = "/kaggle/input/cafa6-protein-embeddings-esm2"

FASTA loader + ID normalization

In [5]:
def read_fasta_to_df(path: str) -> pd.DataFrame:
    ids, seqs = [], []
    cur_id, cur_seq = None, []
    with open(path, "r", encoding="utf-8") as f:
        for raw in f:
            line = raw.strip()
            if not line: 
                continue
            if line.startswith(">"):
                if cur_id is not None:
                    ids.append(cur_id); seqs.append("".join(cur_seq))
                cur_id = line[1:].split()[0]
                cur_seq = []
            else:
                cur_seq.append(line)
    if cur_id is not None:
        ids.append(cur_id); seqs.append("".join(cur_seq))
    return pd.DataFrame({"protein_id": ids, "sequence": seqs})

def normalize_protein_id(x: str) -> str:
    x = str(x).strip().split()[0]
    if "|" in x:
        parts = x.split("|")
        if len(parts) >= 2:
            x = parts[1]
        else:
            x = parts[-1]
    if "." in x:
        left, right = x.rsplit(".", 1)
        if right.isdigit():
            x = left
    return x

train_seq = read_fasta_to_df(TRAIN_FASTA)
test_seq  = read_fasta_to_df(TEST_FASTA)

train_seq["protein_id"] = train_seq["protein_id"].map(normalize_protein_id)
test_seq["protein_id"]  = test_seq["protein_id"].map(normalize_protein_id)

train_terms = pd.read_csv(TRAIN_TERMS, sep="\t")
train_terms = train_terms.rename(columns={"EntryID":"protein_id","term":"go_id"})
train_terms["protein_id"] = train_terms["protein_id"].map(normalize_protein_id)

print(train_seq.shape, test_seq.shape, train_terms.shape)
print(train_terms["aspect"].value_counts())

(82404, 2) (224309, 2) (537027, 3)
aspect
P    250805
C    157770
F    128452
Name: count, dtype: int64


Split train/val (no duplication)

In [6]:
SEED = 42
VAL_FRAC = 0.10

seq_to_rep = train_seq.groupby("sequence")["protein_id"].min().rename("rep_id").reset_index()
train_rep = train_seq.merge(seq_to_rep, on="sequence", how="left")

rep_ids = train_rep["rep_id"].drop_duplicates().values
rng = np.random.default_rng(SEED); rng.shuffle(rep_ids)

n_val = int(len(rep_ids) * VAL_FRAC)
val_rep = set(rep_ids[:n_val])
train_rep_set = set(rep_ids[n_val:])

val_ids   = train_rep.loc[train_rep["rep_id"].isin(val_rep), "protein_id"].unique().tolist()
train_ids = train_rep.loc[train_rep["rep_id"].isin(train_rep_set), "protein_id"].unique().tolist()

print("train:", len(train_ids), "val:", len(val_ids), "overlap:", len(set(train_ids)&set(val_ids)))

train: 74150 val: 8254 overlap: 0


Load embeddings

In [7]:
def find_file(patterns):
    files = []
    for root, _, fnames in os.walk(EMB_DIR):
        for f in fnames:
            files.append(os.path.join(root, f))
    for p in patterns:
        cand = [x for x in files if p in os.path.basename(x).lower()]
        if cand:
            return cand[0]
    return None

print("EMB_DIR listing (top):", os.listdir(EMB_DIR)[:20])

train_emb_path = find_file(["train", "emb", ".npy"]) or find_file(["train", ".parquet"]) or find_file(["train", ".npz"])
test_emb_path  = find_file(["test",  "emb", ".npy"]) or find_file(["test",  ".parquet"]) or find_file(["test",  ".npz"])
id_path_train  = find_file(["train", "id"]) or find_file(["train", "protein"])
id_path_test   = find_file(["test",  "id"]) or find_file(["test",  "protein"])

print("train_emb_path:", train_emb_path)
print("test_emb_path :", test_emb_path)
print("id_path_train :", id_path_train)
print("id_path_test  :", id_path_test)

def load_embeddings(emb_path, id_path=None):
    if emb_path.endswith(".parquet"):
        df = pd.read_parquet(emb_path)
        if "protein_id" in df.columns and "embedding" in df.columns:
            ids = df["protein_id"].astype(str).tolist()
            X = np.vstack(df["embedding"].to_numpy())
            return ids, X
        elif "protein_id" in df.columns:
            ids = df["protein_id"].astype(str).tolist()
            X = df.drop(columns=["protein_id"]).to_numpy()
            return ids, X
        else:
            raise ValueError("Parquet embeddings missing protein_id column.")
    elif emb_path.endswith(".npz"):
        z = np.load(emb_path, allow_pickle=True)
        keyX = "embeddings" if "embeddings" in z else ("X" if "X" in z else None)
        keyI = "protein_id" if "protein_id" in z else ("ids" if "ids" in z else None)
        if keyX is None or keyI is None:
            raise ValueError(f"npz keys: {list(z.keys())}")
        ids = [str(x) for x in z[keyI]]
        X = z[keyX]
        return ids, X
    elif emb_path.endswith(".npy"):
        X = np.load(emb_path, mmap_mode="r")
        if id_path is None:
            raise ValueError("Need an id file to align .npy embeddings.")
        if id_path.endswith(".csv") or id_path.endswith(".tsv"):
            ids = pd.read_csv(id_path, header=None, sep="\t" if id_path.endswith(".tsv") else ",")[0].astype(str).tolist()
        else:
            ids = [str(x) for x in np.load(id_path, allow_pickle=True)]
        return ids, X
    else:
        raise ValueError("Unknown embedding format: " + emb_path)

train_emb_ids, X_train_all = load_embeddings(train_emb_path, id_path_train)
test_emb_ids,  X_test_all  = load_embeddings(test_emb_path,  id_path_test)

train_emb_ids = [normalize_protein_id(x) for x in train_emb_ids]
test_emb_ids  = [normalize_protein_id(x) for x in test_emb_ids]

print("X_train_all:", np.shape(X_train_all), "X_test_all:", np.shape(X_test_all))

EMB_DIR listing (top): ['protein_embeddings.npy', 'protein_ids.csv', 'README.md']
train_emb_path: /kaggle/input/cafa6-protein-embeddings-esm2/protein_embeddings.npy
test_emb_path : /kaggle/input/cafa6-protein-embeddings-esm2/protein_embeddings.npy
id_path_train : /kaggle/input/cafa6-protein-embeddings-esm2/protein_ids.csv
id_path_test  : /kaggle/input/cafa6-protein-embeddings-esm2/protein_ids.csv
X_train_all: (287001, 1280) X_test_all: (287001, 1280)


In [8]:
if len(test_emb_ids) != X_test_all.shape[0]:
    n = min(len(test_emb_ids), X_test_all.shape[0])
    print("Aligning test ids/embeddings to length:", n)
    test_emb_ids = test_emb_ids[:n]
    X_test_all = X_test_all[:n]

test_id_to_row = {pid:i for i, pid in enumerate(test_emb_ids)}

print("after align -> ids:", len(test_emb_ids), "X:", X_test_all.shape[0])

Aligning test ids/embeddings to length: 287001
after align -> ids: 287001 X: 287001


In [9]:
if len(test_emb_ids) == X_test_all.shape[0] + 1 and test_emb_ids[-1] == "A4QQE0":
    test_emb_ids = test_emb_ids[:-1]
    # X_test_all unchanged
test_id_to_row = {pid:i for i, pid in enumerate(test_emb_ids)}

In [10]:
def make_split_embeddings(pids, X_all, id_to_row):
    rows = []
    kept = []
    for pid in pids:
        r = id_to_row.get(pid, None)
        if r is None:
            continue
        rows.append(r)
        kept.append(pid)
    X = np.asarray(X_all[rows], dtype=np.float32)
    return kept, X

In [11]:
test_pids_aligned, X_test = make_split_embeddings(
    test_seq["protein_id"].tolist(), X_test_all, test_id_to_row
)

In [12]:
print("aligned test proteins:", len(test_pids_aligned), "expected:", len(test_seq))
print("missing:", len(test_seq) - len(test_pids_aligned))

aligned test proteins: 224308 expected: 224309
missing: 1


In [13]:
missing = list(set(test_seq["protein_id"]) - set(test_pids_aligned))
missing[:10], len(missing)

(['A4QQE0'], 1)

**clean fallback for just A4QQE0**

In [14]:
top_fallback = {}
for asp in ["BP","MF","CC"]:
    s = train_terms[(train_terms["protein_id"].isin(train_ids)) & (train_terms["aspect"]==asp)]["go_id"].value_counts()
    top_fallback[asp] = s.head(200).index.tolist()
    print(asp, "fallback terms:", len(top_fallback[asp]))

BP fallback terms: 0
MF fallback terms: 0
CC fallback terms: 0


debugging BP fallback terms: 0
MF fallback terms: 0
CC fallback terms: 0

In [15]:
print("len(train_ids):", len(train_ids))
print("train_terms proteins:", train_terms["protein_id"].nunique())

matches = train_terms["protein_id"].isin(set(train_ids)).sum()
print("matches rows:", matches)

print("example train_ids:", train_ids[:5])
print("example train_terms ids:", train_terms["protein_id"].head(5).tolist())

len(train_ids): 74150
train_terms proteins: 82404
matches rows: 484630
example train_ids: ['A0A0C5B5G6', 'A0JNW5', 'A0JP26', 'A0PK11', 'A1A4S6']
example train_terms ids: ['Q5W0B1', 'Q5W0B1', 'Q5W0B1', 'Q5W0B1', 'Q5W0B1']


**fallback from train_terms**


In [16]:
top_fallback = {}
for asp in ["BP","MF","CC"]:
    s = train_terms[train_terms["aspect"]==asp]["go_id"].value_counts()
    top_fallback[asp] = s.head(200).index.tolist()
    print(asp, "fallback terms:", len(top_fallback[asp]))

BP fallback terms: 0
MF fallback terms: 0
CC fallback terms: 0


In [17]:
print(train_terms["aspect"].value_counts().head(20))
print("unique aspects:", sorted(train_terms["aspect"].astype(str).unique())[:20])

aspect
P    250805
C    157770
F    128452
Name: count, dtype: int64
unique aspects: ['C', 'F', 'P']


In [18]:
train_terms["aspect"] = train_terms["aspect"].astype(str).str.strip().str.upper()

aspect_map = {
    "BPO": "BP", "BP": "BP", "P": "BP", "BIOLOGICAL_PROCESS": "BP",
    "MFO": "MF", "MF": "MF", "F": "MF", "MOLECULAR_FUNCTION": "MF",
    "CCO": "CC", "CC": "CC", "C": "CC", "CELLULAR_COMPONENT": "CC",
}

train_terms["asp_norm"] = train_terms["aspect"].map(aspect_map)
print(train_terms["asp_norm"].value_counts(dropna=False))

asp_norm
BP    250805
CC    157770
MF    128452
Name: count, dtype: int64


In [19]:
top_fallback = {}
for asp in ["BP","MF","CC"]:
    s = train_terms[(train_terms["asp_norm"]==asp)]["go_id"].value_counts()
    top_fallback[asp] = s.head(200).index.tolist()
    print(asp, "fallback terms:", len(top_fallback[asp]), "top1:", top_fallback[asp][0] if top_fallback[asp] else None)

BP fallback terms: 200 top1: GO:0045944
MF fallback terms: 200 top1: GO:0005515
CC fallback terms: 200 top1: GO:0005634


Label vocab per aspect + multi-hot targets

In [20]:
ASPECTS = ["BP", "MF", "CC"]

terms_train_split = train_terms[train_terms["protein_id"].isin(train_ids)].copy()

go_vocab = {}
go2i = {}

for asp in ASPECTS:
    vocab = sorted(terms_train_split.loc[terms_train_split["asp_norm"]==asp, "go_id"].unique())
    go_vocab[asp] = vocab
    go2i[asp] = {go:i for i,go in enumerate(vocab)}
    print(asp, "labels:", len(vocab))

def pid2labels_for_aspect(pids_set, asp):
    df = train_terms[(train_terms["protein_id"].isin(pids_set)) & (train_terms["asp_norm"]==asp)][["protein_id","go_id"]].copy()
    df = df[df["go_id"].isin(go2i[asp])]
    df["y"] = df["go_id"].map(go2i[asp]).astype(int)
    return df.groupby("protein_id")["y"].apply(list).to_dict()

train_pid2labels = {asp: pid2labels_for_aspect(set(train_ids), asp) for asp in ASPECTS}
val_pid2labels   = {asp: pid2labels_for_aspect(set(val_ids),   asp) for asp in ASPECTS}

print("example BP labels for a train protein:", next(iter(train_pid2labels["BP"].items())))

BP labels: 16525
MF labels: 6442
CC labels: 2609
example BP labels for a train protein: ('A0A023I7E1', [57])


Align embeddings to split IDs

In [21]:
train_id_to_row = {pid:i for i,pid in enumerate(train_emb_ids)}
test_id_to_row  = {pid:i for i,pid in enumerate(test_emb_ids)}

def make_split_embeddings(pids, X_all, id_to_row):
    rows = [id_to_row[pid] for pid in pids if pid in id_to_row]
    kept = [pid for pid in pids if pid in id_to_row]
    X = np.asarray(X_all[rows], dtype=np.float32)
    return kept, X

train_pids_aligned, X_train = make_split_embeddings(train_ids, X_train_all, train_id_to_row)
val_pids_aligned,   X_val   = make_split_embeddings(val_ids,   X_train_all, train_id_to_row)
test_pids_aligned,  X_test  = make_split_embeddings(test_seq["protein_id"].tolist(), X_test_all, test_id_to_row)

print("aligned train/val/test:", len(train_pids_aligned), len(val_pids_aligned), len(test_pids_aligned))

aligned train/val/test: 74150 8254 224308


Train MLP heads with logits-stable version using binary_cross_entropy_with_logits

In [22]:
import torch.nn.functional as F

class ASLStable(nn.Module):
    def __init__(self, gamma_pos=0.0, gamma_neg=2.0):
        super().__init__()
        self.gp = gamma_pos
        self.gn = gamma_neg

    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")

        p = torch.sigmoid(logits)
        pt = p*targets + (1-p)*(1-targets)

        w = (1-pt).pow(self.gp*targets + self.gn*(1-targets))
        return (w*bce).mean()

In [23]:
loss_fn = ASLStable(gamma_pos=0.0, gamma_neg=2.0)

**Training each aspect**

In [24]:
class EmbDataset(Dataset):
    def __init__(self, X, pids, pid2labels):
        self.X = X
        self.pids = pids
        self.pid2labels = pid2labels

    def __len__(self):
        return len(self.pids)

    def __getitem__(self, i):
        pid = self.pids[i]
        labs = self.pid2labels.get(pid, [])
        return self.X[i], labs

In [25]:
def collate_emb(batch, n_labels):
    xs, labs_list = zip(*batch)
    Xb = torch.tensor(np.stack(xs), dtype=torch.float32)
    Yb = torch.zeros((len(xs), n_labels), dtype=torch.float32)
    for i, labs in enumerate(labs_list):
        if labs:
            Yb[i, labs] = 1.0
    return Xb, Yb

In [26]:
class ASLStable(nn.Module):
    def __init__(self, gamma_pos=0.0, gamma_neg=2.0):
        super().__init__()
        self.gp = gamma_pos
        self.gn = gamma_neg
    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        p = torch.sigmoid(logits)
        pt = p*targets + (1-p)*(1-targets)
        w = (1-pt).pow(self.gp*targets + self.gn*(1-targets))
        return (w*bce).mean()

class EmbDataset(Dataset):
    def __init__(self, X, pids, pid2labels):
        self.X = X
        self.pids = pids
        self.pid2labels = pid2labels
    def __len__(self): return len(self.pids)
    def __getitem__(self, i):
        pid = self.pids[i]
        labs = self.pid2labels.get(pid, [])
        return self.X[i], labs

def collate_emb(batch, n_labels):
    xs, labs_list = zip(*batch)
    Xb = torch.tensor(np.stack(xs), dtype=torch.float32)
    Yb = torch.zeros((len(xs), n_labels), dtype=torch.float32)
    for i, labs in enumerate(labs_list):
        if labs:
            Yb[i, labs] = 1.0
    return Xb, Yb

class MLP(nn.Module):
    def __init__(self, d_in, d_hid, n_out):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, d_hid),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(d_hid, n_out),
        )
    def forward(self, x): return self.net(x)

D = X_train.shape[1]
print("embedding dim:", D)

embedding dim: 1280


In [27]:
SEED = 42
VAL_FRAC = 0.10

EPOCHS = 6
BATCH = 1024
LR = 1e-3
NUM_WORKERS = 2

TOPK_SUB = 200
DEFAULT_THR = 0.20
OUT_PATH = "/kaggle/working/submission.tsv"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

np.random.seed(SEED)
torch.manual_seed(SEED)
if device.type == "cuda":
    torch.cuda.manual_seed_all(SEED)

device: cuda


In [28]:
from torch.amp import GradScaler

scaler = GradScaler("cuda", enabled=(device.type == "cuda"))
print("scaler enabled:", scaler.is_enabled())

scaler enabled: True


In [29]:
models = {}
for asp in ASPECTS:
    nlab = len(go_vocab[asp])

    train_ds = EmbDataset(X_train, train_pids_aligned, train_pid2labels[asp])
    val_ds   = EmbDataset(X_val,   val_pids_aligned,   val_pid2labels[asp])

    train_loader = DataLoader(
        train_ds, batch_size=BATCH, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=True,
        collate_fn=lambda b, nlab=nlab: collate_emb(b, nlab)
    )
    val_loader = DataLoader(
        val_ds, batch_size=BATCH, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=True,
        collate_fn=lambda b, nlab=nlab: collate_emb(b, nlab)
    )

    model = MLP(D, 2048, nlab).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)

    # IMPORTANT: use the stable loss you created
    loss_fn = ASLStable(gamma_pos=0.0, gamma_neg=2.0)

    for ep in range(1, EPOCHS + 1):
        model.train()
        for Xb, Yb in train_loader:
            Xb = Xb.to(device, non_blocking=True)
            Yb = Yb.to(device, non_blocking=True)

            opt.zero_grad(set_to_none=True)
            with autocast("cuda", enabled=(device.type == "cuda")):
                logits = model(Xb)
                loss = loss_fn(logits, Yb)

            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()

        print(f"{asp} epoch {ep} done")

    models[asp] = model

BP epoch 1 done
BP epoch 2 done
BP epoch 3 done
BP epoch 4 done
BP epoch 5 done
BP epoch 6 done
MF epoch 1 done
MF epoch 2 done
MF epoch 3 done
MF epoch 4 done
MF epoch 5 done
MF epoch 6 done
CC epoch 1 done
CC epoch 2 done
CC epoch 3 done
CC epoch 4 done
CC epoch 5 done
CC epoch 6 done


Validation thresholds (per aspect) + NaN check

In [30]:
import numpy as np
import torch

def build_multihot(pids, pid2labels, n_labels):
    Y = np.zeros((len(pids), n_labels), dtype=np.uint8)
    for i, pid in enumerate(pids):
        labs = pid2labels.get(pid, [])
        if labs:
            Y[i, labs] = 1
    return Y

def fmax_safe(Y_true, Y_score, thresholds=np.linspace(0.01, 0.99, 99)):
    best_f, best_t = -1.0, 0.20
    for t in thresholds:
        Y_pred = (Y_score >= t).astype(np.uint8)
        tp = (Y_pred & Y_true).sum()
        fp = (Y_pred & (1 - Y_true)).sum()
        fn = ((1 - Y_pred) & Y_true).sum()
        p = tp / (tp + fp + 1e-9)
        r = tp / (tp + fn + 1e-9)
        f = 2 * p * r / (p + r + 1e-9)
        if f > best_f:
            best_f, best_t = float(f), float(t)
    return best_f, best_t

thr_aspect = {}

Xv = torch.tensor(X_val, dtype=torch.float32)
for asp in ASPECTS:
    model = models[asp].to(device).eval()
    nlab = len(go_vocab[asp])
    Y_true = build_multihot(val_pids_aligned, val_pid2labels[asp], nlab)

    outs = []
    bs = 4096
    with torch.no_grad():
        for i in range(0, Xv.shape[0], bs):
            xb = Xv[i:i+bs].to(device, non_blocking=True)
            prob = torch.sigmoid(model(xb)).float().cpu().numpy()
            outs.append(prob)
    Y_score = np.vstack(outs).astype(np.float32)

    print(asp, "Y_score:", "min", float(np.nanmin(Y_score)), "mean", float(np.nanmean(Y_score)), "max", float(np.nanmax(Y_score)))
    if np.isnan(Y_score).any():
        raise ValueError(f"{asp}: NaNs detected in Y_score (do NOT proceed)")

    best_f, thr = fmax_safe(Y_true, Y_score)
    thr_aspect[asp] = thr
    print(asp, "best_f:", best_f, "thr:", thr)

thr_aspect

BP Y_score: min 1.8556773284217343e-05 mean 0.024377863854169846 max 0.23563911020755768
BP best_f: 0.022524132558921703 thr: 0.16
MF Y_score: min 2.193452928622719e-05 mean 0.021803369745612144 max 0.5758545398712158
MF best_f: 0.31774170191521717 thr: 0.51
CC Y_score: min 0.00010279876732965931 mean 0.03290458396077156 max 0.38439497351646423
CC best_f: 0.18993007526541386 thr: 0.29000000000000004


{'BP': 0.16, 'MF': 0.51, 'CC': 0.29000000000000004}

Predict TEST + write submission.tsv 

In [31]:
Xt = torch.tensor(X_test, dtype=torch.float32)

test_pid_to_row = {pid:i for i,pid in enumerate(test_pids_aligned)}
missing_ids = list(set(test_seq["protein_id"]) - set(test_pids_aligned))

written = 0
with open(OUT_PATH, "w", encoding="utf-8") as f:
    for asp in ASPECTS:
        model = models[asp].eval()
        nlab = len(go_vocab[asp])
        i2go = go_vocab[asp]
        thr = float(thr_aspect.get(asp, DEFAULT_THR))

        bs = 4096
        with torch.no_grad():
            for i in range(0, Xt.shape[0], bs):
                xb = Xt[i:i+bs].to(device, non_blocking=True)
                prob = torch.sigmoid(model(xb)).float().cpu()  # [B, nlab]

                k = min(TOPK_SUB, nlab)
                sc, idx = torch.topk(prob, k=k, dim=1)
                sc = sc.numpy()
                idx = idx.numpy()

                for r in range(sc.shape[0]):
                    pid = test_pids_aligned[i + r]
                    for j in range(sc.shape[1]):
                        s = float(sc[r, j])
                        if s < thr:
                            continue
                        go = i2go[int(idx[r, j])]
                        f.write(f"{pid}\t{go}\t{s:.6f}\n")
                        written += 1

    # fallback for missing protein (A4QQE0)
    for pid in missing_ids:
        for asp in ASPECTS:
            for go in top_fallback[asp]:
                f.write(f"{pid}\t{go}\t{DEFAULT_THR:.6f}\n")
                written += 1

print("Wrote:", OUT_PATH, "| rows:", written, "| missing fallback:", missing_ids[:5])

Wrote: /kaggle/working/submission.tsv | rows: 2441419 | missing fallback: ['A4QQE0']


In [32]:
df = pd.read_csv(OUT_PATH, sep="\t", header=None, names=["protein_id","go_id","score"])
print("rows:", len(df))
print("unique proteins:", df["protein_id"].nunique(), "expected test:", len(test_seq))
print("unique GO:", df["go_id"].nunique())
print("avg rows/protein:", df.groupby("protein_id").size().mean())
print(df.head())

rows: 2441419
unique proteins: 224309 expected test: 224309
unique GO: 615
avg rows/protein: 10.884177629965807
   protein_id       go_id     score
0  A0A0C5B5G6  GO:0045893  0.199449
1  A0A0C5B5G6  GO:0045944  0.196795
2  A0A0C5B5G6  GO:0006355  0.172480
3  A0A0C5B5G6  GO:0006357  0.172165
4  A0A0C5B5G6  GO:0045892  0.169362


In [33]:
df = pd.read_csv(OUT_PATH, sep="\t", header=None, names=["protein_id","go_id","score"])

print("nan scores:", df["score"].isna().sum())
print("score min/max:", df["score"].min(), df["score"].max())
print("bad GO format:", (~df["go_id"].str.match(r"^GO:\d{7}$")).sum())

nan scores: 0
score min/max: 0.16 0.586168
bad GO format: 0


In [34]:
import os
print(os.listdir("/kaggle/input/cafa-6-protein-function-prediction"))

['sample_submission.tsv', 'IA.tsv', 'Test', 'Train']


In [35]:
import pandas as pd

SAMPLE_PATH = "/kaggle/input/cafa-6-protein-function-prediction/sample_submission.tsv"

samp = pd.read_csv(SAMPLE_PATH, header=None, names=["protein_id","go_id","score"])
print(samp.head())
print("shape:", samp.shape)

                                          protein_id  \
0                      A0A0C5B5G6\tGO:0000001\t0.123   
1                      A0A0C5B5G6\tGO:0000002\t0.456   
2  A0A0C5B5G6\tText\t0.123\tRegulates insulin sen...   
3  A0A0C5B5G6\tText\t0.456\tInhibits the folate c...   
4  A0A0C5B5G6\tText\t0.456\tand the activation of...   

                                               go_id  score  
0                                                NaN    NaN  
1                                                NaN    NaN  
2                                                NaN    NaN  
3   thereby reducing de novo purine biosynthesis ...    NaN  
4                                                NaN    NaN  
shape: (20004, 3)


In [36]:
SUB_PATH = "/kaggle/working/submission.tsv"
sub = pd.read_csv(SUB_PATH, sep="\t", header=None, names=["protein_id","go_id","score"])

sub_ids = set(sub["protein_id"].astype(str))
samp_ids = set(samp["protein_id"].astype(str))

print("submission unique ids:", len(sub_ids))
print("sample unique ids    :", len(samp_ids))
print("overlap ids          :", len(sub_ids & samp_ids))
print("missing from sub     :", len(samp_ids - sub_ids))
print("extra in sub         :", len(sub_ids - samp_ids))

print("example missing:", list(samp_ids - sub_ids)[:5])
print("example extra  :", list(sub_ids - samp_ids)[:5])

submission unique ids: 224309
sample unique ids    : 20004
overlap ids          : 0
missing from sub     : 20004
extra in sub         : 224309
example missing: ['O15084\tGO:0000002\t0.456', 'Q9NYQ6\tGO:0000001\t0.123', 'Q9UN37\tGO:0000001\t0.123', 'Q9Y6L6\tGO:0000001\t0.123', 'O14519\tGO:0000002\t0.456']
example extra  : ['P0A6P9', 'Q85G11', 'Q8N957', 'Q8VZ42', 'Q66GV6']


In [37]:
import re
import pandas as pd

SAMPLE_PATH = "/kaggle/input/cafa-6-protein-function-prediction/sample_submission.tsv"
GO_RE = re.compile(r"^GO:\d{7}$")

samp_raw = pd.read_csv(
    SAMPLE_PATH,
    sep="\t",
    header=None,
    engine="python",
    on_bad_lines="skip"   # <-- key fix
)

samp_raw = samp_raw.iloc[:, :3]
samp_raw.columns = ["protein_id", "go_id", "score"]

samp = samp_raw[samp_raw["go_id"].astype(str).str.match(GO_RE)].copy()

samp["protein_id"] = samp["protein_id"].astype(str)
samp["go_id"] = samp["go_id"].astype(str)
samp["score"] = pd.to_numeric(samp["score"], errors="coerce").fillna(0.0)

print("sample rows:", len(samp),
      "unique proteins:", samp["protein_id"].nunique(),
      "unique go:", samp["go_id"].nunique())
print(samp.head())

sample rows: 20000 unique proteins: 10000 unique go: 2
   protein_id       go_id  score
0  A0A0C5B5G6  GO:0000001  0.123
1  A0A0C5B5G6  GO:0000002  0.456
2  A0A1B0GTW7  GO:0000001  0.123
3  A0A1B0GTW7  GO:0000002  0.456
4      A0JNW5  GO:0000001  0.123


In [38]:
SUB_PATH = "/kaggle/working/submission.tsv"
sub = pd.read_csv(SUB_PATH, sep="\t", header=None, names=["protein_id","go_id","score"])
sub["protein_id"] = sub["protein_id"].astype(str)

sub_ids  = set(sub["protein_id"])
samp_ids = set(samp["protein_id"])

print("submission unique ids:", len(sub_ids))
print("sample unique ids    :", len(samp_ids))
print("overlap ids          :", len(sub_ids & samp_ids))
print("missing from sub     :", len(samp_ids - sub_ids))
print("extra in sub         :", len(sub_ids - samp_ids))
print("example missing:", list(samp_ids - sub_ids)[:5])
print("example extra  :", list(sub_ids - samp_ids)[:5])

submission unique ids: 224309
sample unique ids    : 10000
overlap ids          : 10000
missing from sub     : 0
extra in sub         : 214309
example missing: []
example extra  : ['P0A6P9', 'Q85G11', 'Q8N957', 'Q8VZ42', 'Q66GV6']


In [39]:
SUB_PATH = "/kaggle/working/submission.tsv"
sub = pd.read_csv(SUB_PATH, sep="\t", header=None, names=["protein_id","go_id","score"])
sub["protein_id"] = sub["protein_id"].astype(str)

test_ids = set(test_seq["protein_id"].astype(str))
sub_ids  = set(sub["protein_id"])

print("missing test ids in submission:", len(test_ids - sub_ids))
print("extra ids not in test:", len(sub_ids - test_ids))
print("example missing:", list(test_ids - sub_ids)[:5])
print("example extra:", list(sub_ids - test_ids)[:5])

missing test ids in submission: 0
extra ids not in test: 0
example missing: []
example extra: []


In [40]:
import re, numpy as np

GO_OK = sub["go_id"].astype(str).str.match(r"^GO:\d{7}$")
print("bad GO rows:", (~GO_OK).sum())

sub["score"] = pd.to_numeric(sub["score"], errors="coerce")
print("NaN scores:", sub["score"].isna().sum())
print("score min/max:", float(sub["score"].min()), float(sub["score"].max()))
print("out of [0,1]:", ((sub["score"] < 0) | (sub["score"] > 1)).sum())

bad GO rows: 0
NaN scores: 0
score min/max: 0.16 0.586168
out of [0,1]: 0


In [41]:
dup = sub.duplicated(["protein_id","go_id"]).sum()
print("duplicate (protein_id, go_id) rows:", dup)

# If dup > 0, fix it like this:
sub = sub.groupby(["protein_id","go_id"], as_index=False)["score"].max()
sub.to_csv(SUB_PATH, sep="\t", header=False, index=False)
print("rewrote without duplicates:", SUB_PATH, "rows:", len(sub))

duplicate (protein_id, go_id) rows: 0
rewrote without duplicates: /kaggle/working/submission.tsv rows: 2441419


improving score hopefully

In [42]:
import re
from collections import defaultdict, deque
import numpy as np
import torch

ASPECTS = ["BP", "MF", "CC"]

TOPK_PER_ASP = {"BP": 60, "MF": 60, "CC": 60}
MAX_ROWS_PER_PROT = 250
BATCH_PRED = 2048 

DEFAULT_FALLBACK_SCORE = 0.05
OUT_PATH = "/kaggle/working/submission.tsv"

OBO_PATH = "/kaggle/input/cafa-6-protein-function-prediction/Train/go-basic.obo"

ID_RE  = re.compile(r"^id:\s+(GO:\d{7})")
ISA_RE = re.compile(r"^is_a:\s+(GO:\d{7})")

parents = defaultdict(list)
cur = None
with open(OBO_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        m = ID_RE.match(line)
        if m:
            cur = m.group(1)
            continue
        if cur is not None:
            m = ISA_RE.match(line)
            if m:
                parents[cur].append(m.group(1))

anc_cache = {}
def get_ancestors(go_id: str):
    a = anc_cache.get(go_id)
    if a is not None:
        return a
    out = set()
    dq = deque([go_id])
    while dq:
        x = dq.popleft()
        for p in parents.get(x, []):
            if p not in out:
                out.add(p)
                dq.append(p)
    anc_cache[go_id] = out
    return out

print("GO graph ready. #terms with parents:", len(parents))

Xt = torch.tensor(X_test, dtype=torch.float32)
test_ids = test_pids_aligned

missing_ids = list(set(test_seq["protein_id"]) - set(test_pids_aligned))
print("Missing test proteins (fallback):", missing_ids)

written = 0
with open(OUT_PATH, "w", encoding="utf-8") as f:
    for start in range(0, Xt.shape[0], BATCH_PRED):
        xb = Xt[start:start+BATCH_PRED].to(device, non_blocking=True)
        bsz = xb.shape[0]
        batch_pids = test_ids[start:start+bsz]

        per_prot = [defaultdict(float) for _ in range(bsz)]

        with torch.no_grad():
            for asp in ASPECTS:
                model = models[asp].to(device).eval()
                i2go = go_vocab[asp]
                nlab = len(i2go)
                k = min(TOPK_PER_ASP[asp], nlab)

                prob = torch.sigmoid(model(xb)).float().cpu() 
                sc, idx = torch.topk(prob, k=k, dim=1)
                sc = sc.numpy()
                idx = idx.numpy()

                for r in range(bsz):
                    d = per_prot[r]
                    for j in range(k):
                        go = i2go[int(idx[r, j])]
                        s  = float(sc[r, j])
                        if s > d[go]:
                            d[go] = s

        for r in range(bsz):
            d = per_prot[r]
            items = list(d.items())
            for go, s in items:
                for anc in get_ancestors(go):
                    if s > d[anc]:
                        d[anc] = s

            if len(d) > MAX_ROWS_PER_PROT:
                top = sorted(d.items(), key=lambda x: x[1], reverse=True)[:MAX_ROWS_PER_PROT]
                d.clear()
                d.update(top)

            pid = batch_pids[r]
            for go, s in d.items():
                f.write(f"{pid}\t{go}\t{s:.6f}\n")
            written += len(d)

        if (start // BATCH_PRED) % 10 == 0:
            print(f"batch {start}/{Xt.shape[0]} written_rows={written}")

    for pid in missing_ids:
        for asp in ASPECTS:
            for go in top_fallback[asp][:200]:
                f.write(f"{pid}\t{go}\t{DEFAULT_FALLBACK_SCORE:.6f}\n")
                written += 1

print("DONE. Wrote:", OUT_PATH, "| rows:", written)

bad_go = 0
bad_score = 0
line_count = 0
go_re = re.compile(r"^GO:\d{7}$")

with open(OUT_PATH, "r", encoding="utf-8") as fin:
    for i, line in enumerate(fin):
        line_count += 1
        if i < 5:
            print("sample:", line.strip())
        parts = line.rstrip("\n").split("\t")
        if len(parts) != 3:
            bad_score += 1
            continue
        pid, go, sc = parts
        if not go_re.match(go):
            bad_go += 1
        try:
            s = float(sc)
            if not (0.0 <= s <= 1.0):
                bad_score += 1
        except:
            bad_score += 1

print("lines:", line_count, "| bad_go:", bad_go, "| bad_score/format:", bad_score)

GO graph ready. #terms with parents: 40119
Missing test proteins (fallback): ['A4QQE0']
batch 0/224308 written_rows=512000
batch 20480/224308 written_rows=5632000
batch 40960/224308 written_rows=10752000
batch 61440/224308 written_rows=15872000
batch 81920/224308 written_rows=20992000
batch 102400/224308 written_rows=26112000
batch 122880/224308 written_rows=31232000
batch 143360/224308 written_rows=36352000
batch 163840/224308 written_rows=41472000
batch 184320/224308 written_rows=46592000
batch 204800/224308 written_rows=51712000
DONE. Wrote: /kaggle/working/submission.tsv | rows: 56077600
sample: A0A0C5B5G6	GO:0005515	0.548196
sample: A0A0C5B5G6	GO:0003674	0.548196
sample: A0A0C5B5G6	GO:0005488	0.548196
sample: A0A0C5B5G6	GO:0005634	0.373090
sample: A0A0C5B5G6	GO:0043227	0.373090
lines: 56077600 | bad_go: 0 | bad_score/format: 0


In [43]:
print("have models:", "models" in globals())
print("have go_vocab:", "go_vocab" in globals())
print("have thr_aspect:", "thr_aspect" in globals())
print("have X_test:", "X_test" in globals())
print("have test_pids_aligned:", "test_pids_aligned" in globals())
print("have top_fallback:", "top_fallback" in globals())

have models: True
have go_vocab: True
have thr_aspect: True
have X_test: True
have test_pids_aligned: True
have top_fallback: True
